# Phase 3 Runner Smoke Tests

This notebook is for small, local checks of the Phase 3 runner extraction.

It covers:
- direct imports of the extracted runner modules
- the lazy package entry points in `synthetic_runs.runners`
- a tiny sampled-run smoke test through the extracted sampled runner
- a tiny sensitivity-grid smoke test through the extracted sensitivity runner

Use a kernel/environment that has the RAPID dependencies installed.
The `UNC` environment is the intended one for these checks.


In [7]:
from pathlib import Path
import sys
import shutil
import json
import pandas as pd
import plotly.express as px
from IPython.display import display


def find_repo_root() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path('/Users/6256481/Code/river-hierarchy'),
    ]
    for candidate in candidates:
        if (candidate / 'synthetic_runs' / 'src' / 'synthetic_runs').exists() and (candidate / 'RAPID' / 'src' / 'rapid_tools').exists():
            return candidate.resolve()
    raise RuntimeError('Could not locate repo root.')


ROOT = find_repo_root()
SYN_ROOT = ROOT / 'synthetic_runs'
RAPID_ROOT = ROOT / 'RAPID'
SYN_SRC = SYN_ROOT / 'src'
RAPID_SRC = RAPID_ROOT / 'src'
for path in [SYN_SRC, RAPID_SRC]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

OUT = SYN_ROOT / 'outputs' / 'test_phase_3'
OUT.mkdir(parents=True, exist_ok=True)
SAMPLED_IN = SYN_ROOT / 'outputs' / 'test_phase_1' / 'sampled_small'
SAMPLED_OUT = OUT / 'notebook_sampled_smoke'
SENSITIVITY_RECIPES = Path('/Volumes/PhD/river_hierarchy/output/synthetic_network/synthetic_run_sensitivity/networks_sensitivity.jsonl.gz')
SENSITIVITY_OUT = OUT / 'notebook_sensitivity_smoke'

print('ROOT:', ROOT)
print('OUT:', OUT)
print('SAMPLED_IN exists:', SAMPLED_IN.exists())
print('SENSITIVITY_RECIPES exists:', SENSITIVITY_RECIPES.exists())


ROOT: /Users/6256481/Code/river-hierarchy
OUT: /Users/6256481/Code/river-hierarchy/synthetic_runs/outputs/test_phase_3
SAMPLED_IN exists: True
SENSITIVITY_RECIPES exists: True


In [8]:
from synthetic_runs.runners import run_sampled_realizations, run_sensitivity_grid
from synthetic_runs.runners.sampled import run_sampled_realizations as run_sampled_realizations_direct
from synthetic_runs.runners.sensitivity import run_sensitivity_grid as run_sensitivity_grid_direct


## 1. Runner Import Surface

This checks that the lazy package entry points and the extracted module functions are both importable.


In [9]:
print('package sampled entry point:', run_sampled_realizations)
print('direct sampled entry point:', run_sampled_realizations_direct)
print('package sensitivity entry point:', run_sensitivity_grid)
print('direct sensitivity entry point:', run_sensitivity_grid_direct)


package sampled entry point: <function run_sampled_realizations at 0x16571d760>
direct sampled entry point: <function run_sampled_realizations at 0x16571dee0>
package sensitivity entry point: <function run_sensitivity_grid at 0x16571d940>
direct sensitivity entry point: <function run_sensitivity_grid at 0x16601c680>


## 2. Tiny Sampled-Runner Smoke Test

This runs the extracted sampled runner on the tiny Phase 1 sampled fixture.


In [10]:
if SAMPLED_OUT.exists():
    shutil.rmtree(SAMPLED_OUT)
SAMPLED_OUT.mkdir(parents=True, exist_ok=True)

sampled_results = run_sampled_realizations_direct(
    sampled_dir=SAMPLED_IN,
    out_dir=SAMPLED_OUT,
    n_benchmarks=1,
    n_non_benchmarks=0,
    seed=123,
    use_admissible=False,
    keep_intermediate=False,
    write_netcdf=False,
)

sampled_meta = json.loads((SAMPLED_OUT / 'run_meta_routing.json').read_text())
sampled_kq = pd.read_csv(SAMPLED_OUT / 'k_q_metrics.csv')
sampled_peak = pd.read_csv(SAMPLED_OUT / 'peak_q_metrics.csv')
sampled_q = pd.read_parquet(SAMPLED_OUT / 'q_outlet.parquet')

print('sampled results:')
display(pd.DataFrame(sampled_results))
print('selected network ids:', sampled_meta['selected_network_ids'])
print('single-edge control path:', sampled_meta['single_edge_control_path'])
display(sampled_kq)
display(sampled_peak)
display(sampled_q.head())


print sampled dir: /Users/6256481/Code/river-hierarchy/synthetic_runs/outputs/test_phase_1/sampled_small 


print parquet:  /Users/6256481/Code/river-hierarchy/synthetic_runs/outputs/test_phase_1/sampled_small/summary.parquet


Running networks: 100%|██████████| 2/2 [00:00<00:00, 11.82it/s]

Wrote 2 rows to /Users/6256481/Code/river-hierarchy/synthetic_runs/outputs/test_phase_3/notebook_sampled_smoke/k_q_metrics.csv
Wrote 2 rows to /Users/6256481/Code/river-hierarchy/synthetic_runs/outputs/test_phase_3/notebook_sampled_smoke/peak_q_metrics.csv
sampled results:


,network_id,dt,qout_path
0,0,3210.0,/Users/6256481/Code/river-hierarchy/synthetic_...
1,-1,12550.0,/Users/6256481/Code/river-hierarchy/synthetic_...


selected network ids: [0, -1]
single-edge control path: /Users/6256481/Code/river-hierarchy/synthetic_runs/configs/single_edge_control.json


,network_id,geometry_id,sample_type,K_Q,tau_mean,tau_var,K_dom,n_paths,paths_exceeded,B_node,C_node,source_node,outlet_node,source_note,outlet_note,error
0,0,0,benchmark,13281.5662,13281.5662,0.0,13281.5662,2.0,False,"(500.0, 'main')","(2500.0, 'main')","(0.0, 'main')","(3000.0, 'main')",NaN,NaN,NaN
1,-1,-1,single_edge,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,B->C reach set empty


,network_id,geometry_id,sample_type,reach_id,peak_q,peak_time
0,0,0,benchmark,1,0.161462,478290
1,-1,-1,single_edge,1,0.872373,489450


,network_id,geometry_id,sample_type,time,Q
0,0,0,benchmark,0,0.0
1,0,0,benchmark,3210,0.0
2,0,0,benchmark,6420,0.0
3,0,0,benchmark,9630,0.0
4,0,0,benchmark,12840,0.0


## 3. Tiny Sensitivity-Runner Smoke Test

This runs the extracted sensitivity runner on one preserved structural recipe plus the explicit single-edge control.


In [11]:
if SENSITIVITY_OUT.exists():
    shutil.rmtree(SENSITIVITY_OUT)
SENSITIVITY_OUT.mkdir(parents=True, exist_ok=True)

sensitivity_configs = run_sensitivity_grid_direct(
    recipes_path=SENSITIVITY_RECIPES,
    out_dir=SENSITIVITY_OUT,
    network_ids=[0],
    kb_values=[20.0],
    slope_values=[0.003],
    sinuosity_values=[1.0],
    forcing_hours_values=[14.0],
    fall_hours_values=[16.0],
    peak_values=[1e-5],
    baseflow_values=[4e-8],
    keep_intermediate=False,
    write_netcdf=False,
    max_paths=100,
)

sensitivity_meta = json.loads((SENSITIVITY_OUT / 'run_meta_sensitivity.json').read_text())
sensitivity_manifest = pd.read_csv(SENSITIVITY_OUT / 'grid_manifest.csv')
sensitivity_kq = pd.read_csv(SENSITIVITY_OUT / 'k_q_metrics.csv')
sensitivity_peak = pd.read_csv(SENSITIVITY_OUT / 'peak_q_metrics.csv')
sensitivity_err = pd.read_csv(SENSITIVITY_OUT / 'run_errors.csv')
sensitivity_q = pd.read_parquet(SENSITIVITY_OUT / 'q_outlet.parquet')

print('number of configs:', len(sensitivity_configs))
print('single-edge control path:', sensitivity_meta['single_edge_control_path'])
display(sensitivity_manifest)
display(sensitivity_kq)
display(sensitivity_peak)
display(sensitivity_err)
display(sensitivity_q.head())


Wrote grid manifest: /Users/6256481/Code/river-hierarchy/synthetic_runs/outputs/test_phase_3/notebook_sensitivity_smoke/grid_manifest.csv


Running grid: 100%|██████████| 2/2 [00:00<00:00,  6.80it/s]

Wrote 2 rows to /Users/6256481/Code/river-hierarchy/synthetic_runs/outputs/test_phase_3/notebook_sensitivity_smoke/k_q_metrics.csv
Wrote 2 rows to /Users/6256481/Code/river-hierarchy/synthetic_runs/outputs/test_phase_3/notebook_sensitivity_smoke/peak_q_metrics.csv
Wrote empty error file to /Users/6256481/Code/river-hierarchy/synthetic_runs/outputs/test_phase_3/notebook_sensitivity_smoke/run_errors.csv
number of configs: 2
single-edge control path: /Users/6256481/Code/river-hierarchy/synthetic_runs/configs/single_edge_control.json


,network_id,network_type,kb,S_global,S_local,sinuosity,forcing_hours,fall_hours,peak,baseflow,slope_target,sinuosity_target,grid_id,single_edge_control_path
0,0,loop,20.0,0.003,0.003,1.0,14.0,16.0,0.00001,4.000000e-08,w1,w1,1,/Users/6256481/Code/river-hierarchy/synthetic_...
1,-1,single_edge,20.0,0.003,0.003,1.0,14.0,16.0,0.00001,4.000000e-08,NaN,NaN,2,/Users/6256481/Code/river-hierarchy/synthetic_...


,grid_id,network_id,geometry_id,sample_type,network_type,kb,S_global,S_local,sinuosity,forcing_hours,...,K_dom,n_paths,paths_exceeded,B_node,C_node,source_node,outlet_node,source_note,outlet_note,error
0,1,0,1000001,benchmark,loop,20.0,0.003,0.003,1.0,14.0,...,2761.3076,3.0,False,"(3000.0, 'main')","(9000.0, 'main')","(0.0, 'main')","(12000.0, 'main')",NaN,NaN,NaN
1,2,-1,-1,single_edge,single_edge,20.0,0.003,0.003,1.0,14.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,B->C reach set empty


,grid_id,network_id,geometry_id,sample_type,network_type,kb,S_global,S_local,sinuosity,forcing_hours,fall_hours,peak,baseflow,slope_target,sinuosity_target,reach_id,peak_q,peak_time
0,1,0,1000001,benchmark,loop,20.0,0.003,0.003,1.0,14.0,16.0,0.00001,4.000000e-08,w1,w1,1,20573198.0,484300
1,2,-1,-1,single_edge,single_edge,20.0,0.003,0.003,1.0,14.0,16.0,0.00001,4.000000e-08,NaN,NaN,1,79493384.0,488160


,network_id,network_type,kb,S_global,S_local,sinuosity,forcing_hours,fall_hours,peak,baseflow,slope_target,sinuosity_target,grid_id,single_edge_control_path,stage,error


,grid_id,network_id,geometry_id,sample_type,network_type,time,Q,kb,S_global,S_local,sinuosity,forcing_hours,fall_hours,peak,baseflow,slope_target,sinuosity_target
0,1,0,1000001,benchmark,loop,0,0.0,20.0,0.003,0.003,1.0,14.0,16.0,0.00001,4.000000e-08,w1,w1
1,1,0,1000001,benchmark,loop,1450,0.0,20.0,0.003,0.003,1.0,14.0,16.0,0.00001,4.000000e-08,w1,w1
2,1,0,1000001,benchmark,loop,2900,0.0,20.0,0.003,0.003,1.0,14.0,16.0,0.00001,4.000000e-08,w1,w1
3,1,0,1000001,benchmark,loop,4350,0.0,20.0,0.003,0.003,1.0,14.0,16.0,0.00001,4.000000e-08,w1,w1
4,1,0,1000001,benchmark,loop,5800,0.0,20.0,0.003,0.003,1.0,14.0,16.0,0.00001,4.000000e-08,w1,w1


## 4. Hydrograph Diagnostics

These summaries and plots make it easier to see that the first rows are zero because the forcing starts with an initial zero-flow segment, not because the entire run is zero.


In [12]:
def summarize_q(df, group_cols):
    rows = []
    for keys, sub in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = dict(zip(group_cols, keys))
        row['q_min'] = float(sub['Q'].min())
        row['q_max'] = float(sub['Q'].max())
        row['q_mean'] = float(sub['Q'].mean())
        row['nonzero_count'] = int((sub['Q'] != 0).sum())
        nz = sub.loc[sub['Q'] != 0, 'time']
        row['first_nonzero_time'] = None if len(nz) == 0 else int(nz.iloc[0])
        rows.append(row)
    return pd.DataFrame(rows)

sampled_q_plot = sampled_q.copy()
sampled_q_plot['series_id'] = sampled_q_plot['network_id'].astype(str)
sampled_summary = summarize_q(sampled_q_plot, ['network_id'])
display(sampled_summary)
fig_sampled = px.line(
    sampled_q_plot,
    x='time',
    y='Q',
    color='series_id',
    title='Phase 3 sampled smoke: outlet Q vs time',
    labels={'series_id': 'network_id'},
)
fig_sampled.update_layout(height=420)
fig_sampled.show()

sensitivity_q_plot = sensitivity_q.copy()
sensitivity_q_plot['series_id'] = (
    'grid_'
    + sensitivity_q_plot['grid_id'].astype(str)
    + '_net_'
    + sensitivity_q_plot['network_id'].astype(str)
)
sensitivity_summary = summarize_q(sensitivity_q_plot, ['grid_id', 'network_id', 'network_type'])
display(sensitivity_summary)
fig_sensitivity = px.line(
    sensitivity_q_plot,
    x='time',
    y='Q',
    color='series_id',
    title='Phase 3 sensitivity smoke: outlet Q vs time',
    labels={'series_id': 'grid/network'},
)
fig_sensitivity.update_layout(height=420)
fig_sensitivity.show()


,network_id,q_min,q_max,q_mean,nonzero_count,first_nonzero_time
0,-1,0.0,0.872373,0.199926,54,100400
1,0,0.0,0.137737,0.033405,215,89880


,grid_id,network_id,network_type,q_min,q_max,q_mean,nonzero_count,first_nonzero_time
0,1,0,loop,0.0,19896406.0,1.460753e+06,491,88450
1,2,-1,single_edge,0.0,79493384.0,5.847038e+06,156,94920


## 5. Output Check

These are the main files the extracted runner modules should write in this small smoke test.


In [13]:
sampled_files = sorted(p.name for p in SAMPLED_OUT.iterdir())
sensitivity_files = sorted(p.name for p in SENSITIVITY_OUT.iterdir())

print('sampled output files:')
print(sampled_files)
print()
print('sensitivity output files:')
print(sensitivity_files)


sampled output files:
['k_q_metrics.csv', 'peak_q_metrics.csv', 'q_outlet.parquet', 'run_meta_routing.json']

sensitivity output files:
['edge_velocity_tc.parquet', 'grid_manifest.csv', 'k_q_metrics.csv', 'peak_q_metrics.csv', 'q_outlet.parquet', 'run_errors.csv', 'run_meta_sensitivity.json']
